# Document Graph Demo

This notebook shows how to use `genai-graph` to build a **Document Graph** from a
directory of Markdown files: a `Folder → Document → MarkdownSection` graph that lets
an agent navigate a corpus by table of contents and section text — no embeddings,
no vector search.

## What you'll do
1. Configure a directory and graph output path
2. Run `DocumentGraphFactory` + `ingest_document_graph` to build the graph
3. Query the resulting graph with the navigation tools and raw Cypher
4. Visualize the graph inline

**No LLM needed** — just Markdown files and the graph engine.

> To add LLM-based entity extraction alongside the document graph (e.g. Opportunity,
> Risk, Person nodes), see `genai_graph/kg/factories/markdown_baml_factory.py` and the
> "Next Steps" section at the end of this notebook.


## 1. Configuration

Edit the paths below to point at your documents directory and desired output location.

In [ ]:
import tempfile
from pathlib import Path

# ── Edit these paths ──────────────────────────────────────────────────────
# Directory containing documents to ingest (markdown / text files)
DOCUMENTS_DIR = Path("docs")  # genai-graph docs as a sample corpus

# Where to store the Kuzu graph database
DB_PATH = Path(tempfile.mkdtemp()) / "document_graph.db"
# ─────────────────────────────────────────────────────────────────────────

DOCUMENTS_DIR = DOCUMENTS_DIR.resolve()
print(f"Documents : {DOCUMENTS_DIR}")
print(f"Graph DB  : {DB_PATH}")
print(f"Files found: {len(list(DOCUMENTS_DIR.rglob('*.md')))} markdown files")

: 

## 2. Build the Document Graph

`DocumentGraphFactory` scans the directory and, for each Markdown file, builds:
- One **Folder** node (the source directory)
- One **Document** node (path, filename, size, content hash)
- One **MarkdownSection** node per heading, linked by `HAS_SECTION` (Document → root
  section) and `HAS_SUBSECTION` (section → nested section)

`ingest_document_graph` merges all of this into the graph in one call — no manual
node/relationship wiring needed.


In [ ]:
from genai_graph.kg.backend import KuzuBackend
from genai_graph.kg.document_graph.ingest import ingest_document_graph
from genai_graph.kg.factories.document_graph_factory import DocumentGraphFactory

# 1. Open (or create) the graph database
backend = KuzuBackend()
backend.connect(str(DB_PATH))
print(f"✅ Opened graph DB at {DB_PATH}")

# 2. Initialise the factory and inspect the schema it will create
factory = DocumentGraphFactory(sources=[str(DOCUMENTS_DIR)], include=["*.md"])

schema = factory.build_schema()
print(f"Schema nodes     : {[n.node_class.__name__ for n in schema.nodes]}")
print(f"Schema relations : {[r.name for r in schema.relations]}")

In [ ]:
# 3. Ingest — creates the schema tables, parses each file's heading hierarchy,
#    and MERGEs Folder/Document/MarkdownSection nodes + relationships in one call.
result = ingest_document_graph(backend, factory)

print(f"  Documents processed : {result.documents_processed}")
print(f"  Documents skipped   : {result.documents_skipped}")
print(f"  Sections created    : {result.sections_created}")
print(f"  Relationships       : {result.relationships_created}")
for w in result.warnings:
    print(f"  ⚠ {w}")

## 3. Query the Graph

Use standard Cypher to explore the graph.

In [ ]:
from rich.console import Console
from rich.table import Table

console = Console()


def run_query(cypher: str, title: str = "Results") -> None:
    """Execute a Cypher query and display as a Rich table."""
    try:
        df = backend.execute_get_as_df(cypher, union=True)
        if df.empty:
            console.print(f"[yellow]{title}: no results[/yellow]")
            return
        table = Table(title=f"{title} ({len(df)} rows)")
        for col in df.columns:
            table.add_column(str(col), style="cyan")
        for _, row in df.head(20).iterrows():
            table.add_row(*[str(v) for v in row])
        console.print(table)
        if len(df) > 20:
            console.print(f"[dim]… {len(df) - 20} more rows[/dim]")
    except Exception as exc:
        console.print(f"[red]Query error: {exc}[/red]")

In [ ]:
# Node counts by type
run_query("MATCH (d:Document) RETURN d.filename, d.file_size, d.section_count ORDER BY d.filename", "Documents")

In [ ]:
# Sections per document
run_query(
    """
    MATCH (d:Document)-[:HAS_SECTION|HAS_SUBSECTION*]->(s:MarkdownSection)
    RETURN d.filename AS document, count(s) AS sections
    ORDER BY sections DESC
    """,
    "Sections per Document",
)

In [ ]:
# Table of contents for the first ingested document, using the navigation tools
from genai_graph.kg.query.document_graph_tools import get_document_toc, list_documents

docs = list_documents(backend)
if docs:
    first_doc = docs[0]
    print(f"Table of contents for: {first_doc['filename']}\n")
    for row in get_document_toc(backend, first_doc["markdown_hash"]):
        indent = "  " * max(int(row["level"]) - 1, 0)
        print(f"{indent}- {row['title']} (line {row['line_start']})")
else:
    print("No documents ingested yet.")

## 4. Visualize the Graph

Generate an interactive HTML graph and display it inline.

In [ ]:
import sys
import tempfile
from pathlib import Path

from genai_graph.kg.export.html import generate_html

# Fetch a compact subgraph for visualization (documents + their root section)
cypher = """
MATCH (f:Folder)-[:CONTAINS]->(d:Document)-[:HAS_SECTION]->(s:MarkdownSection)
RETURN f, d, s
LIMIT 50
"""

try:
    html_content = generate_html(
        connection=backend,
        destination_file_path="/tmp/document_graph.html",
        query=cypher,
    )
    _html_path = Path(tempfile.mkdtemp()) / "kg_data.html"
    _html_path.write_text(html_content, encoding="utf-8")
    if "ipykernel" in sys.modules:
        from IPython.display import HTML, display  # noqa: PLC0415

        display(HTML(html_content))
    else:
        print(f"Graph HTML saved: {_html_path}")
except Exception as exc:
    print(f"Graph HTML not available: {exc}")
    print("Use 'cli docgraph build' or 'cli kg view' for a full visualization.")

## 5. Schema Visualization

Display the schema (node types and relationships) as an interactive D3 diagram.

In [ ]:
import sys
import tempfile
from pathlib import Path

schema = factory.build_schema()

try:
    from genai_graph.kg.schema import ResolvedSchema

    resolved = ResolvedSchema.from_graph_schema(schema)
    schema_html = resolved.to_html()
    _schema_path = Path(tempfile.mkdtemp()) / "schema.html"
    _schema_path.write_text(schema_html, encoding="utf-8")
    if "ipykernel" in sys.modules:
        from IPython.display import HTML, display  # noqa: PLC0415

        display(HTML(schema_html))
    else:
        print(f"Schema HTML saved: {_schema_path}")
    print("\nNode types:")
    for node in schema.nodes:
        print(f"  - {node.label}")
    print("\nRelationships:")
    for rel in schema.relations:
        print(f"  - {rel.from_node.label} -[{rel.name}]-> {rel.to_node.label}")
except Exception as exc:
    print(f"Schema visualization not available: {exc}")

## Alternative: Use the CLI

Instead of the manual steps above, use `cli docgraph build` — it markdownizes the
sources first (already-Markdown files are copied through unchanged) and ingests
them in one call:

```bash
# Build (or update) the Document Graph
cli docgraph build ./docs --db ./data/kg/tree.db

# Browse it
cli docgraph list --db ./data/kg/tree.db
cli docgraph toc <filename-or-hash> --db ./data/kg/tree.db
cli docgraph search "workflow" --db ./data/kg/tree.db
cli docgraph tui --db ./data/kg/tree.db
```

Or, for a Markdown corpus you already have and want wired into the generic workflow
engine, run the bundled `document_graph` workflow:

```bash
uv run cli workflow run document_graph --dry-run
uv run cli workflow run document_graph --set sources=[./docs] --set db_path=./data/kg/tree.db
```


## Next Steps: Entity Extraction

To extract structured entities (Opportunity, Risk, Person, …) from the same Markdown
files and merge them into this Document Graph:

1. Define a BAML schema with your domain entities.
2. Subclass `genai_graph.kg.factories.markdown_baml_factory.MarkdownBamlFactory`,
   implementing `build_schema()` (your entity nodes + a `MENTIONS` relation back to
   `Document`, via `DocumentMixin.get_document_schema_elements()`) and
   `extract_from_markdown()` (calls your BAML function).
3. Run it alongside `DocumentGraphFactory` in the same KG — see
   `genai_graph.orchestration.workflow_steps.docgraph_build_step`, which markdownizes
   sources, runs entity factories, and ingests the document graph into one database
   (Document nodes MERGE by content hash, so both share the same node).

See [docs/document-graph.md](../docs/document-graph.md) and the `ekg-atos` project's
`ekg_atos/schema/rainbow_review.py` for a full example.
